# Non-personalized Recommender: Random (Normal Distribution)

This notebook implements a non-personalized random recommender where scores are sampled from a normal distribution around rating means.

Metrics used in this notebook are only:
- Accuracy@K
- NDCG@K
- Diversity
- Personalisation
- Coverage


## 1) Environment setup and robust data loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for c in candidates:
        if (c / 'dataset').exists() and (c / 'EDA').exists():
            return c
    raise FileNotFoundError('Could not locate project root containing dataset/ and EDA/.')

PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / 'dataset'
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

WINE_PATH = DATASET_DIR / 'XWines_Full_100K_wines.csv'
RATING_PATH = DATASET_DIR / 'XWines_Full_21M_ratings.csv'

wines = pd.read_csv(
    WINE_PATH,
    usecols=['WineID', 'WineName', 'Type', 'Country', 'RegionName'],
    dtype={
        'WineID': 'int32',
        'WineName': 'string',
        'Type': 'category',
        'Country': 'category',
        'RegionName': 'string'
    }
)

ratings = pd.read_csv(
    RATING_PATH,
    usecols=['UserID', 'WineID', 'Rating'],
    dtype={'UserID': 'int32', 'WineID': 'int32', 'Rating': 'float32'}
)

wines = wines.drop_duplicates(subset='WineID').copy()
ratings = ratings.drop_duplicates(subset=['UserID', 'WineID']).copy()
ratings = ratings[ratings['WineID'].isin(wines['WineID'])].copy()

print('wines:', wines.shape)
print('ratings:', ratings.shape)
print('users:', ratings['UserID'].nunique())
print('items:', ratings['WineID'].nunique())


## 2) EDA-guided snapshot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ratings['Rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#4C78A8')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

item_counts = ratings['WineID'].value_counts()
item_counts.head(50).reset_index(drop=True).plot(ax=axes[1], color='#F58518')
axes[1].set_title('Top-50 Item Popularity')
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Ratings count')

plt.tight_layout()
plt.show()


## 3) Positive Interactions and Train/Test Split


In [ ]:
POSITIVE_THRESHOLD = 4.0
MIN_POSITIVE_INTERACTIONS = 5
MAX_POSITIVE_INTERACTIONS = 250
MAX_USERS = 5000
SAMPLED_USERS_PATH = RESULTS_DIR / 'nonpers_sampled_users_5000.csv'

positive = ratings[ratings['Rating'] >= POSITIVE_THRESHOLD].copy()
user_counts = positive['UserID'].value_counts()

eligible_users = user_counts[
    (user_counts >= MIN_POSITIVE_INTERACTIONS) &
    (user_counts <= MAX_POSITIVE_INTERACTIONS)
].index
positive = positive[positive['UserID'].isin(eligible_users)].copy()

if SAMPLED_USERS_PATH.exists():
    sampled_users = pd.read_csv(SAMPLED_USERS_PATH)['UserID'].to_numpy()
    sampled_users = np.intersect1d(sampled_users, positive['UserID'].unique())
else:
    rng = np.random.default_rng(RANDOM_STATE)
    sampled_users = rng.choice(
        positive['UserID'].unique(),
        size=min(MAX_USERS, positive['UserID'].nunique()),
        replace=False
    )
    pd.Series(sampled_users, name='UserID').to_csv(SAMPLED_USERS_PATH, index=False)

positive = positive[positive['UserID'].isin(sampled_users)].copy()


def split_train_test_per_user(df, test_fraction=0.2, random_state=42):
    train_parts, test_parts = [], []
    for _, g in df.groupby('UserID'):
        g = g.sample(frac=1.0, random_state=random_state)
        n_test = max(1, int(np.ceil(len(g) * test_fraction)))
        test_g = g.iloc[:n_test]
        train_g = g.iloc[n_test:]
        if len(train_g) == 0:
            continue
        train_parts.append(train_g)
        test_parts.append(test_g)
    return pd.concat(train_parts, ignore_index=True), pd.concat(test_parts, ignore_index=True)


def assign_fold_ids_per_user(df, n_folds=3, random_state=42):
    chunks = []
    for _, g in df.groupby('UserID'):
        g = g.sample(frac=1.0, random_state=random_state).copy()
        g['fold_id'] = np.arange(len(g)) % n_folds
        chunks.append(g)
    return pd.concat(chunks, ignore_index=True)


trainval_pos, test_pos = split_train_test_per_user(positive, test_fraction=0.2, random_state=RANDOM_STATE)
trainval_folds = assign_fold_ids_per_user(trainval_pos, n_folds=3, random_state=RANDOM_STATE)

print('trainval:', trainval_pos.shape)
print('test:', test_pos.shape)


## 4) Metrics used in this notebook

We only use:
- **Accuracy@K**
- **NDCG@K**
- **Diversity@K**
- **Personalisation@10**
- **Coverage**


In [ ]:
def accuracy_at_k(relevant, recommended, k):
    rec_k = recommended[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for x in rec_k if x in relevant)
    return hits / k


def dcg_at_k(relevant, recommended, k):
    score = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            score += 1.0 / np.log2(i + 1)
    return score


def ndcg_at_k(relevant, recommended, k):
    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg_at_k(relevant, recommended, k) / ideal_dcg


In [ ]:
item_meta = wines.set_index('WineID')[['Type', 'Country']].copy()
item_meta['Type'] = item_meta['Type'].astype(str).str.lower().fillna('unknown')
item_meta['Country'] = item_meta['Country'].astype(str).str.lower().fillna('unknown')


def item_signature(wine_id):
    if wine_id not in item_meta.index:
        return {'unknown'}
    row = item_meta.loc[wine_id]
    return {f"type:{row['Type']}", f"country:{row['Country']}"}


item_signatures = {wid: item_signature(wid) for wid in item_meta.index}


def jaccard_distance(a, b):
    inter = len(a & b)
    union = len(a | b)
    if union == 0:
        return 0.0
    return 1.0 - (inter / union)


def intra_list_diversity(recommended):
    if len(recommended) < 2:
        return 0.0
    dists = []
    for i in range(len(recommended)):
        for j in range(i + 1, len(recommended)):
            a = item_signatures.get(recommended[i], {'unknown'})
            b = item_signatures.get(recommended[j], {'unknown'})
            dists.append(jaccard_distance(a, b))
    return float(np.mean(dists)) if dists else 0.0


def personalisation_at_k(recs_by_user, k=10, max_users=300, random_state=42):
    users = list(recs_by_user.keys())
    if len(users) < 2:
        return 0.0

    rng = np.random.default_rng(random_state)
    if len(users) > max_users:
        users = rng.choice(users, size=max_users, replace=False).tolist()

    sets = {u: set(recs_by_user[u][:k]) for u in users}

    sims = []
    for u1, u2 in itertools.combinations(users, 2):
        a, b = sets[u1], sets[u2]
        union = len(a | b)
        sim = (len(a & b) / union) if union > 0 else 0.0
        sims.append(sim)

    if not sims:
        return 0.0
    return float(1.0 - np.mean(sims))


In [ ]:
def evaluate_model(model, recommend_fn, eval_users, relevant_dict, ks=(5, 10, 20)):
    rows = []
    recs_by_user = {}
    max_k = max(ks)
    all_top10_items = []

    for user_id in eval_users:
        relevant = relevant_dict.get(user_id, set())
        if not relevant:
            continue

        recs = recommend_fn(model, user_id, top_k=max_k)
        if len(recs) == 0:
            continue

        recs_by_user[user_id] = recs
        all_top10_items.extend(recs[:10])

        row = {'UserID': user_id}
        for k in ks:
            row[f'Accuracy@{k}'] = accuracy_at_k(relevant, recs, k)
            row[f'NDCG@{k}'] = ndcg_at_k(relevant, recs, k)
            row[f'Diversity@{k}'] = intra_list_diversity(recs[:k])

        rows.append(row)

    eval_df = pd.DataFrame(rows)
    if eval_df.empty:
        return eval_df, {}, recs_by_user

    coverage = len(set(all_top10_items)) / len(model['all_items']) if len(model['all_items']) else 0.0
    pers = personalisation_at_k(recs_by_user, k=10, max_users=300, random_state=RANDOM_STATE)

    extra = {
        'Coverage': coverage,
        'Personalisation@10': pers
    }
    return eval_df, extra, recs_by_user


In [ ]:
def plot_model_diagnostics(eval_df, recs_by_user, model_name, top_k=10):
    if eval_df.empty:
        print(f'No evaluation rows for {model_name}')
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    sns.histplot(eval_df[f'Accuracy@{top_k}'], bins=25, kde=True, ax=axes[0], color='#4C78A8')
    axes[0].set_title(f'{model_name} - Accuracy@{top_k}')

    sns.histplot(eval_df[f'NDCG@{top_k}'], bins=25, kde=True, ax=axes[1], color='#54A24B')
    axes[1].set_title(f'{model_name} - NDCG@{top_k}')

    all_items = []
    for _, recs in recs_by_user.items():
        all_items.extend(recs[:top_k])

    freq = pd.Series(all_items).value_counts() if all_items else pd.Series(dtype='int64')
    if freq.empty:
        axes[2].text(0.5, 0.5, 'No recs', ha='center', va='center')
    else:
        sns.histplot(freq.values, bins=30, kde=False, ax=axes[2], color='#F58518')
    axes[2].set_title(f'{model_name} - Coverage histogram')
    axes[2].set_xlabel('recommendation count per item')

    plt.tight_layout()
    plt.show()


## 5) Random-normal recommender

For each unseen item we sample:
`score_i ~ N(mu_i, sigma_global)`

where `mu_i` is the item mean rating on train (fallback to global mean).


In [ ]:
def fit_random_normal_model(train_df, all_items, seed=42):
    item_mean = train_df.groupby('WineID')['Rating'].mean()
    global_mean = float(train_df['Rating'].mean())
    global_std = float(train_df['Rating'].std(ddof=0))

    train_seen = train_df.groupby('UserID')['WineID'].apply(set).to_dict()
    all_items = np.array(sorted(all_items), dtype=np.int32)

    return {
        'item_mean': item_mean,
        'global_mean': global_mean,
        'global_std': max(global_std, 1e-3),
        'seed': seed,
        'train_seen': train_seen,
        'all_items': all_items,
    }


def recommend_random_normal(model, user_id, top_k=10):
    seen = model['train_seen'].get(user_id, set())
    candidates = [wid for wid in model['all_items'] if wid not in seen]
    if not candidates:
        return []

    means = model['item_mean'].reindex(candidates).fillna(model['global_mean']).to_numpy()
    sigma = model['global_std']

    rng = np.random.default_rng(model['seed'] + int(user_id))
    sampled_scores = rng.normal(loc=means, scale=sigma, size=len(candidates))

    order = np.argsort(sampled_scores)[::-1]
    return [int(candidates[i]) for i in order[:top_k]]


## 6) Final Evaluation on Held-out Test


In [ ]:
final_model = fit_random_normal_model(
    trainval_pos,
    all_items=wines['WineID'].unique(),
    seed=RANDOM_STATE
)

test_relevant = test_pos.groupby('UserID')['WineID'].apply(set).to_dict()
eval_users = sorted(set(final_model['train_seen'].keys()) & set(test_relevant.keys()))

random_eval_df, random_extra, random_recs_by_user = evaluate_model(
    final_model,
    recommend_random_normal,
    eval_users,
    test_relevant,
    ks=(5, 10, 20)
)

random_summary = random_eval_df.drop(columns='UserID').mean().to_dict()
random_summary.update(random_extra)
random_summary = pd.DataFrame([random_summary], index=['RandomNormal'])
display(random_summary)
print('Sampling sigma is fixed to global_std.')


In [ ]:
plot_model_diagnostics(random_eval_df, random_recs_by_user, model_name='RandomNormal', top_k=10)


### Qualitative examples: 3 users

Here we show recommendation lists for 3 different users to showcase different products returned by the random recommender.


In [ ]:
example_users = sorted(random_recs_by_user.keys())[:3]
print('Example users:', example_users)

for user_id in example_users:
    recs = recommend_random_normal(final_model, user_id, top_k=10)
    print(f'\nUser {user_id}')
    display(
        wines[wines['WineID'].isin(recs)][['WineID', 'WineName', 'Type', 'Country']]
        .set_index('WineID')
        .loc[recs]
    )


## 7) Final Output Table (Only Required Metrics)


In [ ]:
ordered_cols = [
    'Accuracy@5', 'Accuracy@10', 'Accuracy@20',
    'NDCG@5', 'NDCG@10', 'NDCG@20',
    'Diversity@5', 'Diversity@10', 'Diversity@20',
    'Personalisation@10', 'Coverage'
]
available_cols = [c for c in ordered_cols if c in random_summary.columns]
random_summary = random_summary[available_cols]
display(random_summary)


In [ ]:
plot_cols = ['Accuracy@10', 'NDCG@10', 'Diversity@10', 'Personalisation@10', 'Coverage']
avail = [c for c in plot_cols if c in random_summary.columns]

if avail:
    fig, axes = plt.subplots(1, len(avail), figsize=(4 * len(avail), 4))
    if len(avail) == 1:
        axes = [axes]
    for ax, col in zip(axes, avail):
        random_summary[col].plot(kind='bar', ax=ax, color='#4C78A8')
        ax.set_title(col)
        ax.set_ylabel('score')
    plt.tight_layout()
    plt.show()


In [ ]:
metrics_path = RESULTS_DIR / 'nonpers_random_metrics.csv'
summary_path = RESULTS_DIR / 'nonpers_random_summary.csv'
random_eval_df.to_csv(metrics_path, index=False)
random_summary.to_csv(summary_path)

print('Saved:')
print('-', metrics_path)
print('-', summary_path)
